#Lab Activity 06

**Learning Vector Quantization (LVQ)** is a prototype-based, **supervised**
classifier trained by competitive learning. It has two layers — an input layer
with one node per feature, and an output layer with one node per prototype.

It is the supervised counterpart of the Self-Organising Map from Lab 04. Both
find a winning node by Euclidean distance, but they use the answer differently:

| | SOM (Lab 04) | LVQ (this lab) |
|---|---|---|
| Learning | Unsupervised — no labels | **Supervised** — labels drive the update |
| The winner | Always moves **towards** the input | Moves towards it **only if the labels match**, otherwise **away** |
| Neighbours | Move too | Only the winner moves |
| Purpose | Map and visualise structure | Classify |

**The algorithm.** Prototypes (the weight vectors) are initialised from training
samples, one per class. Then for each training example:

1. Find the **winning prototype** — the one closest to the example.
2. If the winner's class **matches** the example's label, move it **towards**
   the example. If not, push it **away**.

$$w_j(t+1) = w_j(t) \pm \alpha\,(x_k - w_j(t))$$

The `+` is used when the classification was right and the `-` when it was wrong.
A new example is then labelled with the class of whichever prototype wins.

##Activity 1: Implementation of LVQ in Python

The manual gives an `LVQ` class with a `winner` method and an `update` method,
plus a driver that trains on six 4-bit samples.

**The code as printed does not run, and two further lines are wrong.** All four
problems are listed below, then the corrected version is built and each bug is
demonstrated.

| # | As printed in the manual | Problem | Fix |
|---|---|---|---|
| 1 | `if actual -- j:` | `--` is not a comparison, and `j` (lower case) is never defined — the loop variable is `J`. Python reads this as `actual - (-j)`, so it raises `NameError`. | `if actual == J:` |
| 2 | The `if`/`else` inside `update` | The `if` is indented four spaces further than its `else`, which is an `IndentationError`. | Align them |
| 3 | `if D0 > D1: return 0 else: return 1` | **Returns the prototype that is further away.** If `D0 > D1` then prototype 1 is the closer one, so the winner should be `1`. | `return 0 if D0 < D1 else 1` |
| 4 | `for i in range(len(weights))` inside `update` | `len(weights)` is the number of **prototypes** (2), not the number of **features** (4). Only the first two features of the winning prototype are ever updated. | `for i in range(len(sample))` |

Problems 1 and 2 stop the code running at all. Problems 3 and 4 let it run while
doing the wrong thing, which makes them the more interesting ones — they are
demonstrated at the end of this activity.

###The corrected implementation

In [1]:
import math


class LVQ:

    # Function here computes the winning vector
    # by Euclidean distance
    def winner(self, weights, sample):
        D0 = 0
        D1 = 0

        for i in range(len(sample)):
            D0 = D0 + math.pow((sample[i] - weights[0][i]), 2)
            D1 = D1 + math.pow((sample[i] - weights[1][i]), 2)

        # The winner is the CLOSER prototype, so the smaller distance wins
        if D0 < D1:
            return 0
        else:
            return 1

    # Function here updates the winning vector
    def update(self, weights, sample, J, alpha, actual):
        # range(len(sample)) - one step per feature, not per prototype
        if actual == J:
            # Correct classification: move the winner TOWARDS the sample
            for i in range(len(sample)):
                weights[J][i] = weights[J][i] + alpha * (sample[i] - weights[J][i])
        else:
            # Wrong classification: push the winner AWAY from the sample
            for i in range(len(sample)):
                weights[J][i] = weights[J][i] - alpha * (sample[i] - weights[J][i])


print("LVQ class defined.")

LVQ class defined.


###Running it on the manual's dataset

Six samples of four binary features, in two classes. The first two samples
become the initial prototypes — one per class — and are then removed from the
training set, exactly as the manual describes.

In [2]:
def main():
    # Training Samples ( m, n ) with their class vector
    X = [[0, 0, 1, 1], [1, 0, 0, 0],
         [0, 0, 0, 1], [0, 1, 1, 0],
         [1, 1, 0, 0], [1, 1, 1, 0]]
    Y = [0, 1, 0, 1, 1, 1]
    m, n = len(X), len(X[0])

    # weight initialization ( n, c ): the first sample of each class
    weights = []
    weights.append(X.pop(0))
    weights.append(X.pop(0))

    # Samples used in weight initialization will not be used in training
    m = m - 2
    Y.pop(0)
    Y.pop(0)

    print("Initial prototypes:", weights)
    print("Remaining training samples:", X, "with labels", Y)

    # training
    ob = LVQ()
    epochs = 3
    alpha = 0.1

    for i in range(epochs):
        for j in range(m):
            # Sample selection
            T = X[j]

            # Compute winner
            J = ob.winner(weights, T)

            # Update weights
            ob.update(weights, T, J, alpha, Y[j])

    # classify new input sample
    T = [0, 0, 1, 0]
    J = ob.winner(weights, T)
    print("\nSample T belongs to class :", J)
    print("Trained weights :", [[round(v, 4) for v in w] for w in weights])


main()

Initial prototypes: [[0, 0, 1, 1], [1, 0, 0, 0]]
Remaining training samples: [[0, 0, 0, 1], [0, 1, 1, 0], [1, 1, 0, 0], [1, 1, 1, 0]] with labels [0, 1, 1, 1]

Sample T belongs to class : 0
Trained weights : [[0.0, -0.1791, 0.703, 1.1791], [0.919, 0.5217, 0.3129, 0.0]]


The answer is sensible: `T = [0, 0, 1, 0]` looks much more like prototype 0's
starting pattern `[0, 0, 1, 1]` than prototype 1's `[1, 0, 0, 0]`, and class 0
is what comes back.

Notice too that all four values of both prototypes have moved away from the
integers they started as. That is the signature of a correct `update` — with bug
4 still in place, the last two columns would be untouched.

###Demonstrating the two logic bugs

Bugs 1 and 2 announce themselves by raising an exception. Bugs 3 and 4 are
quieter, so it is worth showing exactly what each one does.

####Bug 3 — the winner was the prototype furthest away

The cleanest test is a sample that **is** one of the prototypes. Its distance to
that prototype is exactly 0, so there is no argument about which should win.

In [3]:
weights = [[0, 0, 1, 1], [1, 0, 0, 0]]
sample = [1, 0, 0, 0]          # identical to prototype 1

D0 = sum((sample[i] - weights[0][i]) ** 2 for i in range(len(sample)))
D1 = sum((sample[i] - weights[1][i]) ** 2 for i in range(len(sample)))

print("The sample is an exact copy of prototype 1.")
print("  distance to prototype 0 (D0):", D0)
print("  distance to prototype 1 (D1):", D1)
print("  so the winner must be       : 1")

# The manual's rule: `if D0 > D1: return 0 else: return 1`
manual_winner = 0 if D0 > D1 else 1
print("\n  the manual's rule returns   :", manual_winner, " <-- the further prototype")
print("  the corrected rule returns  :", LVQ().winner(weights, sample))

The sample is an exact copy of prototype 1.
  distance to prototype 0 (D0): 3
  distance to prototype 1 (D1): 0
  so the winner must be       : 1

  the manual's rule returns   : 0  <-- the further prototype
  the corrected rule returns  : 1


####Bug 4 — only the first two features were ever updated

`len(weights)` is 2 because there are two prototypes. `len(sample)` is 4 because
there are four features. Looping over the wrong one means features 2 and 3 are
never touched, no matter how long training runs.

In [4]:
def update_manual_version(weights, sample, J, alpha, actual):
    """The manual's update, looping over len(weights) instead of len(sample)."""
    if actual == J:
        for i in range(len(weights)):           # 2, not 4
            weights[J][i] = weights[J][i] + alpha * (sample[i] - weights[J][i])
    else:
        for i in range(len(weights)):
            weights[J][i] = weights[J][i] - alpha * (sample[i] - weights[J][i])


sample = [1.0, 1.0, 1.0, 1.0]

buggy = [[0.0, 0.0, 0.0, 0.0], [9.0, 9.0, 9.0, 9.0]]
update_manual_version(buggy, sample, 0, 0.5, 0)

fixed = [[0.0, 0.0, 0.0, 0.0], [9.0, 9.0, 9.0, 9.0]]
LVQ().update(fixed, sample, 0, 0.5, 0)

print("Prototype 0 started as [0.0, 0.0, 0.0, 0.0]")
print("Moving it halfway towards [1.0, 1.0, 1.0, 1.0] should give [0.5]*4")
print()
print("  manual's version  ->", buggy[0], " features 2 and 3 never moved")
print("  corrected version ->", fixed[0])

Prototype 0 started as [0.0, 0.0, 0.0, 0.0]
Moving it halfway towards [1.0, 1.0, 1.0, 1.0] should give [0.5]*4

  manual's version  -> [0.5, 0.5, 0.0, 0.0]  features 2 and 3 never moved
  corrected version -> [0.5, 0.5, 0.5, 0.5]


###Why the manual's output still looked plausible

Running the manual's version on its own six-sample dataset returns
**class 0** for `T` — the same answer the corrected version gives. That is the
trap: with two prototypes, an inverted winner often just relabels which
prototype is which, and on a dataset this small the final answer can come out
the same for the wrong reasons.

The trained weights are not the same, though. The manual's version ends with

```
[[-0.83, -0.95, 1, 1], [1.08, 0.22, 0, 0]]
```

where the `1, 1` and `0, 0` in the last two columns are the original integers,
never updated — bug 4 in plain sight. The corrected version moves all four.

This is why the graded task below implements LVQ from scratch rather than
extending this class: the manual's version is hard-wired to exactly two
prototypes, and LVQ is at its most useful on multi-class problems.

#End!